# Instance-to-Class Mapping Derivation: Report

This notebook analyses the **cost** and **productivity** of the
instance-to-class mapping derivation pipeline (see
`docs/notes/instance_to_class_mappings_plan.md` §18).

It reads two types of artefacts:

1. **Session report JSON** files emitted by
   `derive_class_mappings_from_instances()` with cost, enrichment, and
   derivation metrics.
2. **Enriched instance JSON-LD** files: the instance-level mapping
   files annotated with `@type` and `rdfsolve:classifiedIn` by the
   class index enrichment step (§6.7).

**Sections:**

1. Setup & data loading
2. Cost dashboard (SPARQL queries, timing, IRI expansion)
3. Enrichment analysis (entity coverage, class discovery, missing prefixes)
4. Derivation productivity (compression ratio, confidence, top pairs)
5. Cross-source comparison
6. Iteration support (re-derive with different thresholds)


## 1. Setup & Data Loading

In [ ]:
from __future__ import annotations

import json
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# paths
ROOT = Path("..").resolve()  # repo root when notebook is in notebooks/
REPORTS_DIR = ROOT / "docker" / "mappings" / "class_derived" / ".session_reports"
ENRICHED_DIR = ROOT / "docker" / "mappings"  # enriched .jsonld files live alongside originals
CLASS_DERIVED_DIR = ROOT / "docker" / "mappings" / "class_derived"

# matplotlib style
plt.rcParams.update({
    "figure.dpi": 120,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "font.size": 10,
})
sns.set_palette("muted")
print("Environment ready.")


In [ ]:
# Load all session reports
report_files = sorted(REPORTS_DIR.glob("*.json")) if REPORTS_DIR.exists() else []
print(f"Found {len(report_files)} session report(s) in {REPORTS_DIR}")

reports: list[dict] = []
for rf in report_files:
    with open(rf) as f:
        reports.append(json.load(f))

if not reports:
    print("No session reports found.  Run the derivation pipeline first:")
    print("   rdfsolve instance-match derive --input ... --output ...")
else:
    print(f"\nSources: {[r.get('source_name', '?') for r in reports]}")


In [ ]:
# Flatten reports into a DataFrame for easy comparison
def flatten_report(report: dict) -> dict:
    """Flatten nested session report into a single-level dict."""
    flat = {
        "source_name": report.get("source_name", "unknown"),
        "timestamp": report.get("timestamp", ""),
        "source_mapping_type": report.get("source_mapping_type", ""),
        "endpoint_url": report.get("endpoint_url", ""),
    }
    for section in ("cost", "enrichment", "derivation"):
        for k, v in report.get(section, {}).items():
            if isinstance(v, (int, float, bool, str)):
                flat[f"{section}.{k}"] = v
    return flat

if reports:
    df_reports = pd.DataFrame([flatten_report(r) for r in reports])
    display(df_reports.T)
else:
    df_reports = pd.DataFrame()


## 2. Cost Dashboard

How expensive was the pipeline?  SPARQL query volume, total wall-clock
time, IRI expansion fan-out.

In [ ]:
if reports:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    # 2a. SPARQL queries sent per source
    names = [r.get("source_name", "?") for r in reports]
    queries = [r.get("cost", {}).get("sparql_queries_sent", 0) for r in reports]
    axes[0].barh(names, queries, color="steelblue")
    axes[0].set_xlabel("SPARQL queries")
    axes[0].set_title("Queries per source")

    # 2b. Total SPARQL time per source
    times = [r.get("cost", {}).get("sparql_total_time_s", 0) for r in reports]
    axes[1].barh(names, times, color="darkorange")
    axes[1].set_xlabel("Total time (s)")
    axes[1].set_title("SPARQL wall-clock time")

    # 2c. IRI expansion fan-out
    fanout = [r.get("cost", {}).get("iri_alternatives_mean_per_entity", 0) for r in reports]
    axes[2].barh(names, fanout, color="seagreen")
    axes[2].set_xlabel("Avg. URI forms per entity")
    axes[2].set_title("Bioregistry expansion fan-out")

    fig.suptitle("Cost Dashboard", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No reports to plot.")


In [ ]:
# Cost summary table
if reports:
    cost_cols = [
        "source_name",
        "cost.iris_total",
        "cost.iri_alternatives_total",
        "cost.sparql_queries_sent",
        "cost.sparql_total_time_s",
        "cost.sparql_mean_time_per_query_s",
        "cost.sparql_max_time_per_query_s",
        "cost.sparql_errors",
        "cost.batch_size",
    ]
    available = [c for c in cost_cols if c in df_reports.columns]
    display(df_reports[available])


## 3. Enrichment Analysis

How many entities were found in the LSLOD cloud?  Which prefixes are
missing?  How many classes were discovered per entity?

In [ ]:
if reports:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for i, r in enumerate(reports):
        enr = r.get("enrichment", {})
        name = r.get("source_name", "?")

        # 3a. Entity coverage pie
        found = enr.get("entities_enriched", 0)
        not_found = enr.get("entities_not_found", 0)
        if found or not_found:
            axes[0].pie(
                [found, not_found],
                labels=["Enriched", "Not found"],
                autopct="%1.1f%%",
                colors=["#4CAF50", "#F44336"],
                startangle=90,
            )
            axes[0].set_title(f"Entity coverage: {name}")

        # 3b. Not-found prefixes bar
        nf_prefixes = enr.get("not_found_prefixes", {})
        if nf_prefixes:
            top_nf = dict(sorted(nf_prefixes.items(), key=lambda x: -x[1])[:15])
            axes[1].barh(list(top_nf.keys()), list(top_nf.values()), color="#F44336")
            axes[1].set_xlabel("Entities not found")
            axes[1].set_title(f"Missing prefixes: {name}")

        # 3c. Classes per entity stats
        axes[2].bar(
            ["Mean", "Max"],
            [
                enr.get("classes_per_entity_mean", 0),
                enr.get("classes_per_entity_max", 0),
            ],
            color=["#2196F3", "#FF9800"],
        )
        axes[2].set_ylabel("Classes per entity")
        axes[2].set_title(f"Class discovery: {name}")

    fig.suptitle("Enrichment Analysis", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No reports to plot.")


In [ ]:
# Enrichment summary table
if reports:
    enr_cols = [
        "source_name",
        "enrichment.entities_total",
        "enrichment.entities_enriched",
        "enrichment.entities_not_found",
        "enrichment.entities_not_found_pct",
        "enrichment.classes_added",
        "enrichment.distinct_classes",
        "enrichment.graphs_referenced",
    ]
    available = [c for c in enr_cols if c in df_reports.columns]
    display(df_reports[available])


## 4. Derivation Productivity

How productive was the derivation?  Compression ratio (instances →
class pairs), confidence distribution, predicate breakdown.

In [ ]:
if reports:
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    for r in reports:
        der = r.get("derivation", {})
        name = r.get("source_name", "?")

        # 4a. Compression ratio
        inp = der.get("input_edges", 1)
        out = der.get("output_edges", 0)
        ratio = inp / out if out > 0 else float("inf")
        axes[0].bar([name], [ratio], color="#9C27B0")
        axes[0].set_ylabel("Instance edges / class pairs")
        axes[0].set_title("Compression ratio")

        # 4b. Confidence distribution
        conf_vals = [der.get("confidence_mean", 0), der.get("confidence_median", 0), der.get("confidence_max", 0)]
        axes[1].bar(["Mean", "Median", "Max"], conf_vals, color=["#2196F3", "#4CAF50", "#FF9800"])
        axes[1].set_ylabel("Confidence")
        axes[1].set_ylim(0, 1.1)
        axes[1].set_title(f"Confidence: {name}")

        # 4c. Predicate distribution
        pred_dist = der.get("predicates_distribution", {})
        if pred_dist:
            preds = list(pred_dist.keys())
            counts = list(pred_dist.values())
            axes[2].barh(preds, counts, color="#00BCD4")
            axes[2].set_xlabel("Class pairs")
            axes[2].set_title(f"Predicates: {name}")

    fig.suptitle("Derivation Productivity", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
else:
    print("No reports to plot.")


In [ ]:
# Top class pairs table
if reports:
    for r in reports:
        der = r.get("derivation", {})
        name = r.get("source_name", "?")
        top_pairs = der.get("top_class_pairs", [])
        if top_pairs:
            print(f"\nTop class pairs: {name}")
            df_top = pd.DataFrame(top_pairs)
            display(df_top.head(20))


In [ ]:
# Derivation summary table
if reports:
    der_cols = [
        "source_name",
        "derivation.input_edges",
        "derivation.class_pairs_found",
        "derivation.class_pairs_after_filter",
        "derivation.output_edges",
        "derivation.min_instance_count",
        "derivation.min_confidence",
        "derivation.confidence_mean",
        "derivation.confidence_median",
    ]
    available = [c for c in der_cols if c in df_reports.columns]
    display(df_reports[available])


## 5. Cross-Source Comparison

Compare cost and productivity across all sources that have been
processed (SSSOM vs SeMRA, different SSSOM bundles, etc.).

In [ ]:
if len(reports) >= 2:
    fig, axes = plt.subplots(1, 2, figsize=(12, 5))

    names = [r.get("source_name", "?") for r in reports]

    # 5a. Cost vs productivity scatter
    cost_time = [r.get("cost", {}).get("sparql_total_time_s", 0) for r in reports]
    output_edges = [r.get("derivation", {}).get("output_edges", 0) for r in reports]
    axes[0].scatter(cost_time, output_edges, s=100, c="steelblue")
    for i, name in enumerate(names):
        axes[0].annotate(name, (cost_time[i], output_edges[i]), fontsize=8)
    axes[0].set_xlabel("SPARQL time (s)")
    axes[0].set_ylabel("Class pairs derived")
    axes[0].set_title("Cost vs Productivity")

    # 5b. Entity coverage comparison
    enriched_pct = [
        100 - r.get("enrichment", {}).get("entities_not_found_pct", 0)
        for r in reports
    ]
    axes[1].barh(names, enriched_pct, color="#4CAF50")
    axes[1].set_xlabel("Entity coverage (%)")
    axes[1].set_xlim(0, 100)
    axes[1].set_title("LSLOD Entity Coverage")

    fig.suptitle("Cross-Source Comparison", fontsize=13, y=1.02)
    plt.tight_layout()
    plt.show()
elif len(reports) == 1:
    print("Only 1 source. Run more sources for cross-source comparison.")
else:
    print("No reports to compare.")


## 6. Iteration Support

The enriched JSON-LD is the primary artefact.  We can **re-derive**
class-level mappings with different `min_instance_count` /
`min_confidence` thresholds without re-querying QLever.

This section simulates different threshold combinations to find the
optimal balance between coverage and precision.

In [ ]:
# ── 6a. Load enriched JSON-LD and extract class pair evidence ──
# This cell demonstrates the structure; actual re-derivation would
# use rdfsolve.class_derivation.derive_class_mappings() with the
# ClassIndex loaded from cache.

# TODO: Once the derivation engine is implemented, add:
#   from rdfsolve.class_index import load_class_index
#   from rdfsolve.class_derivation import derive_class_mappings
#   from rdfsolve.mapping_models.core import Mapping
#
#   index = load_class_index("docker/mappings/.class_index_cache.json")
#   mapping = Mapping.from_jsonld(enriched_path)
#   for threshold in [1, 5, 10, 50, 100]:
#       pairs = derive_class_mappings(mapping.edges, index, min_instance_count=threshold)
#       print(f"min_instances={threshold:>4d}  →  {len(pairs)} class pairs")

print("⏳ Iteration support will be available after Phase 2 implementation.")

In [ ]:
# ── 6b. Threshold sensitivity plot (from session reports) ──
# If multiple session reports exist with different thresholds,
# plot output_edges vs min_instance_count.

if reports:
    threshold_data = []
    for r in reports:
        der = r.get("derivation", {})
        threshold_data.append({
            "source": r.get("source_name", "?"),
            "min_instance_count": der.get("min_instance_count", 1),
            "min_confidence": der.get("min_confidence", 0),
            "class_pairs": der.get("output_edges", 0),
        })
    df_thresh = pd.DataFrame(threshold_data)
    print("Threshold settings across runs:")
    display(df_thresh)
else:
    print("No reports available.")